# Radix-2 and Bluestein FFT — Usage and Reference
O(N log N) FFT, forward with negative sign and without normalisation; IFFT with positive sign and division by N. The notebook uses the Release assemblies, not the 0.2.0 packages.
The closed-form result for [1,2,3,4] is [10,-2+2i,-2,-2-2i]: a reference independent of the DFT implementation.

In [ ]:
#load "../shared/MathemaSharp.csx"
using System; using System.Linq; using System.Numerics;
using MathemaSharp.Core;
void Require(bool value,string label) { if(!value) throw new Exception(label); Console.WriteLine(label+": OK"); }
var x=new Complex<double>[] {new(1,0),new(2,0),new(3,0),new(4,0)};
var expected=new Complex<double>[] {new(10,0),new(-2,2),new(-2,0),new(-2,-2)};
var spectrum=Signals.Fft<double>(x);
Require(spectrum.Zip(expected).All(p=>double.Hypot(p.First.Real-p.Second.Real,p.First.Imaginary-p.Second.Imaginary)<1e-13),"closed_form_oracle");
Require(Signals.Ifft<double>(spectrum).Zip(x).All(p=>double.Hypot(p.First.Real-p.Second.Real,p.First.Imaginary-p.Second.Imaginary)<1e-13),"round_trip");
Console.WriteLine(string.Join(", ",spectrum));

closed_form_oracle: OK
round_trip: OK
<10; 0>, <-2; 2>, <-2; 0>, <-1.9999999999999998; -2>

In [ ]:
void Check<T>() where T:IFloatingPointIeee754<T>,IMinMaxValue<T> {
 var a=new Complex<T>[] {new(T.One,T.Zero),new(T.CreateChecked(2),T.Zero),new(T.CreateChecked(3),T.Zero),new(T.CreateChecked(4),T.Zero)};
 var b=Signals.Fft<T>(a);
 var tolerance=(T.BitIncrement(T.One)-T.One)*T.CreateChecked(128);
 Require(T.Abs(b[1].Real-T.CreateChecked(-2))<=tolerance && T.Abs(b[1].Imaginary-T.CreateChecked(2))<=tolerance,typeof(T).Name+"_oracle");
}
Check<Half>(); Check<float>(); Check<double>(); Check<Decimal32>(); Check<Decimal64>(); Check<Decimal128>();
var delta=Decimal128.Parse("1e-25");
var d=Signals.Fft<Decimal128>(new Complex<Decimal128>[] {new(Decimal128.One,Decimal128.Zero),new(delta,Decimal128.Zero)});
Require(d[0].Real-Decimal128.One==delta,"decimal128_increment");

Half_oracle: OK
Single_oracle: OK
Double_oracle: OK
Decimal32_oracle: OK
Decimal64_oracle: OK
Decimal128_oracle: OK
decimal128_increment: OK

## Reference and limits
| API | Contract |
|---|---|
| Fft<T> | Input Complex<T>, IFloatingPointIeee754<T> and IMinMaxValue<T> |
| Ifft<T> | 1/N normalisation, does not modify the input |
| Length | Any length, within the limits of memory and of the representability of T |
| Values | Finite inputs; non-finite results raise ArithmeticException |

Twiddles and accumulation are computed in T. Universal accuracy is not guaranteed, nor is the absence of intermediate overflow. A small set of oracles does not certify large transforms. Bluestein uses a radix-2 convolution with padding >= 2N-1. SIMD and benchmarks are not included.
[API catalogue](12-api-reference.ipynb).

## Bluestein: prime length 7
The impulse has a unit spectrum: independent analytic reference.

In [ ]:
var prime = new Complex<double>[] { new(1,0), new(0,0), new(0,0), new(0,0), new(0,0), new(0,0), new(0,0) };
var primeSpectrum = Signals.Fft<double>(prime);
Require(primeSpectrum.All(v => double.Hypot(v.Real-1,v.Imaginary)<1e-12), "bluestein_prime_oracle");
var primeBack = Signals.Ifft<double>(primeSpectrum);
Require(primeBack.Zip(prime).All(p => double.Hypot(p.First.Real-p.Second.Real,p.First.Imaginary-p.Second.Imaginary)<1e-12), "bluestein_prime_roundtrip");

bluestein_prime_oracle: OK
bluestein_prime_roundtrip: OK